In [ ]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
from tqdm import tqdm

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science', 'no-latex'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_id = "06"

traces = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

s = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_stimuli.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

coordinates = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_coordinates.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

traces = traces.read().result()
s = s.read().result()
coordinates = coordinates.read().result()

f = h5py.File(f'/{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
names = [(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)]

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx


reference_anat = scipy.io.loadmat(f'/{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

# mask_idx_list = [106] # tectum neuropil
# mask_idx_list = [65] # pretectum
mask_idx_list = [108] # torus longitudinalis [almost all neurons that end here come from pretectum and tectum neuropil]

def get_mask_and_valid_coordinate_i(mask, mask_idx_list, jc, ir, mask_shape, coordinates):
  for mask_idx in mask_idx_list:
    start_idx = jc[mask_idx]
    end_idx = jc[mask_idx + 1]
    for i in ir[start_idx:end_idx]:
      i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
      mask[i_x, i_y, i_z] = 1
  valid_coordinate_i = []
  for i, c in enumerate(coordinates):
    if mask[c.astype(int)[0], c.astype(int)[1], c.astype(int)[2]]:
      valid_coordinate_i.append(i)
  valid_coordinate_i = np.array(valid_coordinate_i)
  return mask, valid_coordinate_i

mask, valid_coordinate_i = get_mask_and_valid_coordinate_i(mask, mask_idx_list, jc, ir, mask_shape, coordinates)
valid_coordinate_i.shape

Try sbi with neural trace data as-is

In [ ]:
import torch
from sbi.utils import BoxUniform
from sbi.inference import NPE

_ = torch.manual_seed(0)

In [ ]:
valid_coordinate_i_h = np.mean(traces[:, valid_coordinate_i], 0) > 0.15

# valid_coordinate_i_h_l = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] < 300)
# valid_coordinate_i_h_r = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] > 300)

# valid_coordinate_i[valid_coordinate_i_h][valid_coordinate_i_h_l]

traces_selected = traces[:, valid_coordinate_i[valid_coordinate_i_h]]
# traces_selected = traces[:, valid_coordinate_i_h]
traces_selected.shape

In [ ]:
context = 256
n_d = context
horizon = 16
n_neurons = traces_selected.shape[-1]
traces_selected.shape[0]-context-horizon
trace_data_x = np.stack([traces_selected[i:i+context] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_theta = np.stack([traces_selected[i+context+horizon] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context)
trace_data_theta = trace_data_theta.ravel()

In [ ]:
trace_data_x_train = torch.tensor(trace_data_x[::10])
trace_data_theta_train = torch.tensor(trace_data_theta[::10, None])
trace_data_x_train.shape, trace_data_theta_train.shape

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(n_d), high=4.0 * torch.ones(n_d))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

Test on example traces

In [ ]:
# selected_ix = [50765, 50772, 50787, 50797, 50802, 51057, 54002, 54044, 54360, 54420, 54466, 54485, 54592, 54622, 54751, 54762, 54765, 54787, 54793, 54795, 54812, 54818, 55189, 55221, 55313, 55374, 55430, 57621, 58133, 58299, 58305, 58365, 58495, 58513, 58540, 58568, 58587, 58621, 58691, 58752, 58758, 58787, 58801, 58815, 59076, 59105, 59162, 59183, 59198, 59203, 59226, 59252, 59256, 59286, 59287, 59337, 59743, 62071, 62128, 62179, 62212, 62304, 62330, 62331, 62332, 62337, 62539, 62545, 62584, 62634, 62980, 63061, 63105, 63106, 63140, 63162, 63197, 63215, 63259, 63282, 63374, 63502, 63507, 65802, 65882, 65984, 65999, 66017, 66101, 66125, 66131, 66149, 66228, 66229, 66252, 66267, 66271, 66385, 66398, 66431, 66593, 66976, 67137, 69409, 69415, 69478, 69549, 69646, 69691, 69702, 69819, 69839, 69898, 69978, 70103, 70126, 70143, 70151, 70366, 70428, 70523, 70591, 72675, 72719, 72913, 72989, 73041, 73045, 73051, 73063, 73075, 73140, 73259, 73261, 73292, 73402, 73406, 73543, 73640, 73770, 74311, 76433, 76515, 76543, 76640, 76648, 76692, 76742, 76900, 77719, 79635, 79722, 79810, 79875, 80012, 80077, 80209, 80285, 83107, 83199, 85955, 86031, 86069, 86128] # tectum neuropil
# selected_ix = np.random.choice(np.arange(0, 90000), size=30, replace=False)
selected_ix = np.arange(0, valid_coordinate_i_h.shape[0])[valid_coordinate_i_h]
valid_coordinate_i_h = np.mean(traces[:, valid_coordinate_i], 0) > 0.15
selected_ix = valid_coordinate_i[valid_coordinate_i_h]
len(selected_ix)

In [ ]:
n_ix = 15
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_ix[:n_ix])):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(traces[:, neuron_ix][j:j+context])
        theta_hat = posterior.sample((200,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces[:500+context, neuron_ix], 'k')
    ax.plot(np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)
fig.suptitle(f"Example traces and predictions, context {context}, horizon {horizon}, TL", fontsize=14, fontname="Arial")
plt.tight_layout()
plt.savefig(f'/Users/s/Documents/larvae/predictions_c{context}_h{horizon}_TL_active.png', bbox_inches='tight', dpi=500)


Now, we train on activity of the torus longitudinalis (TL) (where stimulus-locking is not as strong as in the pretectum (PT) and tectum neuropil (TN)), but connect the highly active TL neurons to the highly active PT and TN neurons, hoping that the conditioning works.

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx


reference_anat = scipy.io.loadmat(f'/{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

# mask_idx_list = [106] # tectum neuropil
# mask_idx_list = [65] # pretectum
mask_idx_list = [108] # torus longitudinalis [almost all neurons that end here come from pretectum and tectum neuropil]
for mask_idx in mask_idx_list:
  start_idx = jc[mask_idx]
  end_idx = jc[mask_idx + 1]
  for i in ir[start_idx:end_idx]:
    i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
    mask[i_x, i_y, i_z] = 1


valid_coordinate_i = []
for i, c in enumerate(coordinates):
  if mask[c.astype(int)[0], c.astype(int)[1], c.astype(int)[2]]:
    valid_coordinate_i.append(i)
valid_coordinate_i = np.array(valid_coordinate_i)

In [ ]:
valid_coordinate_i.shape

Try sbi with neural trace data as-is

In [ ]:
import torch
from sbi.utils import BoxUniform
from sbi.inference import NPE

_ = torch.manual_seed(0)

In [ ]:
traces_selected = traces[:, valid_coordinate_i]
# traces_selected = traces

In [ ]:
valid_coordinate_i_h = np.mean(traces_selected, 0) > 0.13

valid_coordinate_i_h_l = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] < 300)
valid_coordinate_i_h_r = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] > 300)

valid_coordinate_i[valid_coordinate_i_h][valid_coordinate_i_h_l]

In [ ]:
context = 4
n_d = context
horizon = 16
n_neurons = traces_selected.shape[-1]
traces_selected.shape[0]-context-horizon
trace_data_x = np.stack([traces_selected[i:i+context] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_theta = np.stack([traces_selected[i+context+horizon] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context)
trace_data_theta = trace_data_theta.ravel()

In [ ]:
trace_data_x_train = torch.tensor(trace_data_x[::30])
trace_data_theta_train = torch.tensor(trace_data_theta[::30, None])
trace_data_x_train.shape, trace_data_theta_train.shape

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(n_d), high=4.0 * torch.ones(n_d))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

In [ ]:
valid_coordinate_i_h = np.mean(traces[:, valid_coordinate_i], 0) > 0.13

Test on example traces

In [ ]:
# selected_ix = [50765, 50772, 50787, 50797, 50802, 51057, 54002, 54044, 54360, 54420, 54466, 54485, 54592, 54622, 54751, 54762, 54765, 54787, 54793, 54795, 54812, 54818, 55189, 55221, 55313, 55374, 55430, 57621, 58133, 58299, 58305, 58365, 58495, 58513, 58540, 58568, 58587, 58621, 58691, 58752, 58758, 58787, 58801, 58815, 59076, 59105, 59162, 59183, 59198, 59203, 59226, 59252, 59256, 59286, 59287, 59337, 59743, 62071, 62128, 62179, 62212, 62304, 62330, 62331, 62332, 62337, 62539, 62545, 62584, 62634, 62980, 63061, 63105, 63106, 63140, 63162, 63197, 63215, 63259, 63282, 63374, 63502, 63507, 65802, 65882, 65984, 65999, 66017, 66101, 66125, 66131, 66149, 66228, 66229, 66252, 66267, 66271, 66385, 66398, 66431, 66593, 66976, 67137, 69409, 69415, 69478, 69549, 69646, 69691, 69702, 69819, 69839, 69898, 69978, 70103, 70126, 70143, 70151, 70366, 70428, 70523, 70591, 72675, 72719, 72913, 72989, 73041, 73045, 73051, 73063, 73075, 73140, 73259, 73261, 73292, 73402, 73406, 73543, 73640, 73770, 74311, 76433, 76515, 76543, 76640, 76648, 76692, 76742, 76900, 77719, 79635, 79722, 79810, 79875, 80012, 80077, 80209, 80285, 83107, 83199, 85955, 86031, 86069, 86128] # tectum neuropil
selected_ix = valid_coordinate_i[valid_coordinate_i_h]
len(selected_ix)

In [ ]:
n_ix = 20
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_ix[:n_ix])):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(traces[:, neuron_ix][j:j+context])
        theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces[:500+context, neuron_ix], 'k')
    ax.plot(np.arange(context, theta_hat_i.shape[0]+context), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context, theta_hat_i.shape[0]+context),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)

plt.tight_layout()